## 11. Δυναμική Εξερεύνηση του Ιστού

<!-- book-intro-cell -->
### Εισαγωγή από το βιβλίο

Μέχρι τώρα επικεντρωθήκαμε στην ανάλυση μεμονωμένων ιστοσελίδων· πώς να ανακτούμε το περιεχόμενό τους, να εντοπίζουμε στοιχεία στην HTML και πώς να εξάγουμε χρήσιμη πληροφορία. Ωστόσο, ο Ιστός δεν αποτελείται από απομονωμένα έγγραφα, αλλά από ένα εκτεταμένο δίκτυο συνδεδεμένων σελίδων.

Κάθε σελίδα περιέχει συνδέσμους προς άλλες σελίδες, οι οποίες με τη σειρά τους οδηγούν σε νέες. Αυτό το είδαμε ήδη σε προηγούμενα κεφάλαια. Αν εκμεταλλευτούμε αυτή τη δομή, μπορούμε να περάσουμε από την ανάλυση μιας σελίδας στη συστηματική εξερεύνηση πολλών σελίδων. Η διαδικασία αυτή ονομάζεται crawling.

Σε αντίθεση με το scraping, όπου το ενδιαφέρον μας περιορίζεται σε μια συγκεκριμένη σελίδα, το crawling εισάγει μια επαναληπτική διάσταση, όπου συλλέγουμε συνδέσμους, επισκεπτόμαστε νέες σελίδες και επαναλαμβάνουμε τη διαδικασία. Με τον τρόπο αυτό, ο Ιστός μπορεί να ιδωθεί ως ένας γράφος, τον οποίο εξερευνούμε σταδιακά.


In [ ]:
import re

pattern = re.compile(r"^(/wiki/)[^:]*$")

In [ ]:
import requests
from bs4 import BeautifulSoup
import re
from urllib.parse import urljoin

header = {"User-Agent": "CrawlBase/1.0"}
base_url = "https://en.wikipedia.org"
url = base_url + "/wiki/Data_science"

response = requests.get(url, headers=header)
soup = BeautifulSoup(response.text, "html.parser")

# Regex για φιλτράρισμα Wikipedia άρθρων
pattern = re.compile(r"^(/wiki/)[^:]*$")

max_pages_visited = 10
visited = set()

for link in soup.find_all("a", href=True):
    href = link.get("href")

    # Φιλτράρουμε μόνο valid wiki links
    if not pattern.match(href):
        continue

    # Μετατροπή σε πλήρες URL
    next_url = urljoin(base_url, href)

    # Έλεγχος μνήμης
    if next_url in visited:
        print("Visited already:", next_url)
        continue

    visited.add(next_url)

    print("Visiting:", next_url)
    max_pages_visited -= 1

    if max_pages_visited == 0:
        break

    try:
        r = requests.get(next_url, headers=header)
        print("  Status:", r.status_code)
    except:
        print("  Failed")

Visiting: https://en.wikipedia.org/wiki/Main_Page
  Status: 200
Visited already: https://en.wikipedia.org/wiki/Main_Page
Visiting: https://en.wikipedia.org/wiki/Data_science
  Status: 200
Visited already: https://en.wikipedia.org/wiki/Data_science
Visited already: https://en.wikipedia.org/wiki/Data_science
Visiting: https://en.wikipedia.org/wiki/Information_science
  Status: 200
Visiting: https://en.wikipedia.org/wiki/Computer_science
  Status: 200
Visiting: https://en.wikipedia.org/wiki/Comet_NEOWISE
  Status: 200
Visiting: https://en.wikipedia.org/wiki/Astronomical_survey
  Status: 200
Visiting: https://en.wikipedia.org/wiki/Space_telescope
  Status: 200
Visiting: https://en.wikipedia.org/wiki/Wide-field_Infrared_Survey_Explorer
  Status: 200
Visiting: https://en.wikipedia.org/wiki/Interdisciplinary
  Status: 200
Visiting: https://en.wikipedia.org/wiki/Statistics


### Στρατηγικές αναζήτησης (BFS)

In [ ]:
import requests
from bs4 import BeautifulSoup
import re
from urllib.parse import urljoin
from collections import deque

header = {"User-Agent": "CrawlBase/1.0"}
base_url = "https://en.wikipedia.org"
start_url = base_url + "/wiki/Data_science"

pattern = re.compile(r"^(/wiki/)[^:]*$")

visited = set()
queue = deque([start_url])

max_pages = 10

while queue and len(visited) < max_pages:
    url = queue.popleft()

    if url in visited:
        continue

    print("Visiting:", url)
    visited.add(url)

    try:
        response = requests.get(url, headers=header)
        soup = BeautifulSoup(response.text, "html.parser")

        for link in soup.find_all("a", href=True):
            href = link.get("href")

            if not pattern.match(href):
                continue

            next_url = urljoin(base_url, href)

            if next_url not in visited:
                queue.append(next_url)

    except:
        print("Failed:", url)

Visiting: https://en.wikipedia.org/wiki/Data_science
Visiting: https://en.wikipedia.org/wiki/Main_Page
Visiting: https://en.wikipedia.org/wiki/Information_science
Visiting: https://en.wikipedia.org/wiki/Computer_science
Visiting: https://en.wikipedia.org/wiki/Comet_NEOWISE
Visiting: https://en.wikipedia.org/wiki/Astronomical_survey
Visiting: https://en.wikipedia.org/wiki/Space_telescope
Visiting: https://en.wikipedia.org/wiki/Wide-field_Infrared_Survey_Explorer
Visiting: https://en.wikipedia.org/wiki/Interdisciplinary
Visiting: https://en.wikipedia.org/wiki/Statistics


In [ ]:
import requests
from bs4 import BeautifulSoup
import re
from urllib.parse import urljoin

header = {"User-Agent": "CrawlBase/1.0"}
base_url = "https://en.wikipedia.org"

pattern = re.compile(r"^(/wiki/)[^:]*$")

visited = set()

def dfs(url, max_pages=10):
    if len(visited) >= max_pages:
        return

    if url in visited:
        return

    print("Visiting:", url)
    visited.add(url)

    try:
        response = requests.get(url, headers=header)
        soup = BeautifulSoup(response.text, "html.parser")

        for link in soup.find_all("a", href=True):
            href = link.get("href")

            if not pattern.match(href):
                continue

            next_url = urljoin(base_url, href)

            dfs(next_url, max_pages)

    except:
        print("Failed:", url)

start_url = base_url + "/wiki/Data_science"
dfs(start_url)

Visiting: https://en.wikipedia.org/wiki/Data_science
Visiting: https://en.wikipedia.org/wiki/Main_Page
Visiting: https://en.wikipedia.org/wiki/Wikipedia
Visiting: https://en.wikipedia.org/wiki/English_Wikipedia
Visiting: https://en.wikipedia.org/wiki/Simple_English_Wikipedia
Visiting: https://en.wikipedia.org/wiki/Northern_S%C3%A1mi_Wikipedia
Visiting: https://en.wikipedia.org/wiki/Internet_encyclopedia_project
Visiting: https://en.wikipedia.org/wiki/Online_encyclopedia
Visiting: https://en.wikipedia.org/wiki/Encyclopedia
Visiting: https://en.wikipedia.org/wiki/Encyclopedia_(disambiguation)


### Περιορισμοί

```Python
# μέγιστο πληθος σελίδων
max_pages = 100
if len(visited) >= max_pages:
    break

# μέγιστο βάθος
from collections import deque
queue = deque([(start_url, 0)])
max_depth = 2

# [...]
while queue:
    url, depth = queue.popleft()

    # έλεγχος μέγιστου βάθους
    if depth > max_depth:
        continue

# [...]

# πρόσθεση νέων URLs
queue.append((next_url, depth + 1))
```

In [ ]:
visited = set()
queue = deque([(start_url, 0)])

max_pages = 20
max_depth = 2

while queue and len(visited) < max_pages:
    url, depth = queue.popleft()

    if url in visited or depth > max_depth:
        continue

    print("Visiting:", url, "depth:", depth)
    visited.add(url)

In [ ]:
import urllib.robotparser

rp = urllib.robotparser.RobotFileParser()
rp.set_url("https://en.wikipedia.org/robots.txt")
rp.read()

print(rp.can_fetch("*", "https://en.wikipedia.org/wiki/Data_science"))

False


In [ ]:
import time

time.sleep(1)  # 1 δευτερόλεπτο μεταξύ αιτημάτων

### APIs

In [ ]:
import requests

url = "https://en.wikipedia.org/w/api.php"

params = {
    "action": "query",
    "format": "json",
    "prop": "extracts",
    "exintro": True,
    "explaintext": True,
    "titles": "Data science"
}

response = requests.get(url, params=params, headers={"User-Agent": "ResearchBot/1.0"})
data = response.json()
pages = data["query"]["pages"]
for page_id, page in pages.items():
    print(page["title"])
    print(page["extract"][:300])

Data science
Data science is an interdisciplinary academic field that uses statistics, scientific computing, scientific methods, processing, scientific visualization, algorithms, and systems to extract or extrapolate knowledge from potentially noisy, structured, or unstructured data. 
Data science plays a critic


In [ ]:
results = []
for title in ["Data science", "Machine learning", "Artificial intelligence"]:
    params["titles"] = title
    response = requests.get(url, params=params, headers={"User-Agent": "ResearchBot/1.0"})
    data = response.json()
    pages = data["query"]["pages"]
    for page_id, page in pages.items():
        results.append({
            "title": page["title"],
            "text": page["extract"]
        })
print(results[:2])

[{'title': 'Data science', 'text': 'Data science is an interdisciplinary academic field that uses statistics, scientific computing, scientific methods, processing, scientific visualization, algorithms, and systems to extract or extrapolate knowledge from potentially noisy, structured, or unstructured data. \nData science plays a critical role in modern decision-making by enabling organizations to extract actionable insights from large and complex datasets.\nData science also integrates domain knowledge from the underlying application domain (e.g., natural sciences, information technology, and medicine). Data science is multifaceted and can be described as a science, a research paradigm, a research method, a discipline, a workflow, and a profession.\nData science is "a concept to unify statistics, data analysis, informatics, and their related methods" to "understand and analyze actual phenomena" with data. It uses techniques and theories drawn from many fields within the context of math

In [ ]:
# search API
params = {
    "action": "query",
    "format": "json",
    "list": "search",
    "srsearch": "data science",
    "srlimit": 5
}
response = requests.get(url, params=params, headers={"User-Agent": "ResearchBot/1.0"})
data = response.json()
for result in data["query"]["search"]:
    print(result["title"])

Data science
List of data science software
Data
Biomedical data science
Master in Data Science


  0%|          | 0/10 [00:00<?, ?it/s]

(5000, 21)